In [1]:
!pip install -q datasets tokenizers

In [ ]:
import os
os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

!pip install -q datasets tokenizers huggingface_hub numpy tqdm

from huggingface_hub import login
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

login()  # Paste your HF Write token

# 1. Stream 50,000 English & 50,000 Bengali stories for final tokenizer training
en_iter = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
en_tok_samples = [r["text"].strip() for _, r in zip(range(50000), en_iter) if r.get("text")]

bn_iter = load_dataset(
    "Regional-TinyStories/bangla-generated_4o-mini_2M",
    revision="refs/convert/parquet",
    split="train",
    streaming=True,
)
bn_tok_samples = [str(r["story"]).strip() for _, r in zip(range(50000), bn_iter) if r.get("story")]

# 2. Train 16,384-vocab Unicode BPE (Metaspace -> Punctuation, byte_fallback=True)
tokenizer = Tokenizer(models.BPE(byte_fallback=True))
tokenizer.normalizer = normalizers.NFC()
tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Metaspace(),
    pre_tokenizers.Punctuation(),
])
tokenizer.decoder = decoders.Sequence([
    decoders.Metaspace(),
    decoders.ByteFallback(),
])

trainer = trainers.BpeTrainer(
    vocab_size=16384,
    min_frequency=2,
    special_tokens=["<|endoftext|>", "<|pad|>"],
)

def tok_iterator():
    for e, b in zip(en_tok_samples, bn_tok_samples):
        yield e
        yield b

tokenizer.train_from_iterator(tok_iterator(), trainer=trainer)
os.makedirs("packed_data", exist_ok=True)
tokenizer.save("packed_data/tokenizer.json")

# 3. Verify fertility & lossless round-trip decode
for name, samples in [("English", en_tok_samples[-500:]), ("Bengali", bn_tok_samples[-500:])]:
    words = sum(len(t.split()) for t in samples)
    toks = sum(len(tokenizer.encode(t).ids) for t in samples)
    n_bytes = sum(len(t.encode("utf-8")) for t in samples)
    print(f"{name:8s} | Tokens/Word: {toks/words:.2f} | Bytes/Token: {n_bytes/toks:.2f}")

sample_bn = bn_tok_samples[0][:100]
enc = tokenizer.encode(sample_bn)
dec = tokenizer.decode(enc.ids)
print("\nBengali Tokens :", enc.tokens[:18])
print("Round-trip OK? :", dec.strip() == sample_bn.strip())

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Regional-TinyStories/bangla-generated_4o-mini_2M/resolve/2c7fcbfcc318238e43cfc030d52bac42ba6943f2/default/train/0000.parquet
Retrying in 1s [Retry 1/5].


English  | Tokens/Word: 1.24 | Bytes/Token: 4.08
Bengali  | Tokens/Word: 1.26 | Bytes/Token: 11.62

Bengali Tokens : ['▁একদিন', ',', '▁বনভূমির', '▁মধ্যে', '▁এক', '▁ছোট্ট', '▁জঙ্গলে', '▁একটি', '▁চিরসবুজ', '▁গাছ', '▁ছিল', '।', '▁গাছের', '▁নিচে', '▁একটি', '▁ছোট', '▁পুকুর', '▁ছিল']
Round-trip OK? : True


In [ ]:
import json
import numpy as np
from tqdm.auto import tqdm

EOS_ID = tokenizer.token_to_id("<|endoftext|>")
assert EOS_ID == 0 and tokenizer.get_vocab_size() <= 65535

VAL_TARGET = 2_000_000      # 2M held-out validation tokens per language
TRAIN_TARGET = 200_000_000  # 200M training tokens per language
BATCH_DOCS = 2000

def pack_language(dataset_name, text_col, prefix, revision=None):
    kwargs = {"split": "train", "streaming": True}
    if revision:
        kwargs["revision"] = revision
    stream = load_dataset(dataset_name, **kwargs)

    val_path = f"packed_data/{prefix}_val.bin"
    train_path = f"packed_data/{prefix}_train.bin"

    val_tokens, val_bytes = 0, 0
    train_tokens, train_bytes = 0, 0
    batch = []

    pbar = tqdm(total=VAL_TARGET + TRAIN_TARGET, unit="tok", unit_scale=True, desc=f"Packing {prefix.upper()}")

    with open(val_path, "wb") as f_val, open(train_path, "wb") as f_train:
        for row in stream:
            txt = str(row.get(text_col) or "").strip()
            if not txt:
                continue
            batch.append(txt)

            if len(batch) < BATCH_DOCS:
                continue

            encs = tokenizer.encode_batch(batch)
            for txt_i, enc in zip(batch, encs):
                ids = enc.ids + [EOS_ID]
                arr = np.asarray(ids, dtype=np.uint16)
                n_tok = len(arr)
                n_b = len(txt_i.encode("utf-8")) + 1

                if val_tokens < VAL_TARGET:
                    arr.tofile(f_val)
                    val_tokens += n_tok
                    val_bytes += n_b
                elif train_tokens < TRAIN_TARGET:
                    arr.tofile(f_train)
                    train_tokens += n_tok
                    train_bytes += n_b
                else:
                    break
                pbar.update(n_tok)

            batch.clear()
            if train_tokens >= TRAIN_TARGET:
                break

    pbar.close()
    print(f"[{prefix.upper()}] Val: {val_tokens:,} tok ({val_bytes:,} bytes) | Train: {train_tokens:,} tok")
    return {
        "val_tokens": val_tokens,
        "val_bytes": val_bytes,
        "train_tokens": train_tokens,
        "train_bytes": train_bytes,
    }

meta = {
    "vocab_size": tokenizer.get_vocab_size(),
    "eos_id": EOS_ID,
    "dtype": "uint16",
    "en": pack_language("roneneldan/TinyStories", "text", "en"),
    "bn": pack_language(
        "Regional-TinyStories/bangla-generated_4o-mini_2M",
        "story",
        "bn",
        revision="refs/convert/parquet",
    ),
}

with open("packed_data/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved packed_data/meta.json:")
print(json.dumps(meta, indent=2))

Packing EN:   0%|          | 0.00/202M [00:00<?, ?tok/s]

[EN] Val: 2,000,153 tok (8,294,960 bytes) | Train: 200,000,102 tok


Packing BN:   0%|          | 0.00/202M [00:00<?, ?tok/s]

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Regional-TinyStories/bangla-generated_4o-mini_2M/resolve/2c7fcbfcc318238e43cfc030d52bac42ba6943f2/default/train/0000.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Regional-TinyStories/bangla-generated_4o-mini_2M/resolve/2c7fcbfcc318238e43cfc030d52bac42ba6943f2/default/train/0000.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Regional-TinyStories/bangla-generated_4o-mini_2M/resolve/2c7fcbfcc318238e43cfc030d52bac42ba6943f2/default/train/0000.parquet
Retrying in 2s [Retry 2/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Regional-TinyStories/bangla-generated_4o-mini_2M/resolve/2c7fcbfcc318238e43cfc030d52bac42ba6943f2/default/train/0000.parquet
Retrying in 4s [Retry 3/5].
'The read operation time

[BN] Val: 2,000,085 tok (23,122,207 bytes) | Train: 200,000,141 tok

Saved packed_data/meta.json:
{
  "vocab_size": 16384,
  "eos_id": 0,
  "dtype": "uint16",
  "en": {
    "val_tokens": 2000153,
    "val_bytes": 8294960,
    "train_tokens": 200000102,
    "train_bytes": 824816416
  },
  "bn": {
    "val_tokens": 2000085,
    "val_bytes": 23122207,
    "train_tokens": 200000141,
    "train_bytes": 2307842010
  }
}


In [9]:
import torch.nn as nn
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # Always compute norm in fp32 to prevent fp16 overflow/NaN
        x_f32 = x.float()
        norm = x_f32 * torch.rsqrt(x_f32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(x.dtype)


def precompute_rope(dim: int, max_seq_len: int, theta: float = 10000.0, device="cuda"):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device, dtype=torch.float32) / dim))
    t = torch.arange(max_seq_len, device=device, dtype=torch.float32)
    angles = torch.outer(t, freqs)
    return torch.cos(angles)[None, None, :, :], torch.sin(angles)[None, None, :, :]


def apply_rope(x, cos, sin):
    # Compute rotation in fp32 for stability
    x_f32 = x.float()
    x1, x2 = x_f32[..., ::2], x_f32[..., 1::2]
    out = torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2)
    return out.to(x.dtype)


class GQAAttention(nn.Module):
    def __init__(self, d_model=384, n_heads=6, n_kv_heads=2):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.rep = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q, cos[:, :, :T, :], sin[:, :, :T, :])
        k = apply_rope(k, cos[:, :, :T, :], sin[:, :, :T, :])

        k = k.repeat_interleave(self.rep, dim=1)
        v = v.repeat_interleave(self.rep, dim=1)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).contiguous().view(B, T, C))


class SwiGLU(nn.Module):
    def __init__(self, d_model=384, d_ff=1024):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class TransformerBlock(nn.Module):
    def __init__(self, d_model=384, n_heads=6, n_kv_heads=2, d_ff=1024):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads)
        self.norm2 = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.mlp(self.norm2(x))
        return x


class BilingualSLM(nn.Module):
    def __init__(self, vocab_size=16384, d_model=384, n_layers=8, n_heads=6, n_kv_heads=2, d_ff=1024, max_seq_len=512):
        super().__init__()
        self.config = dict(
            vocab_size=vocab_size, d_model=d_model, n_layers=n_layers,
            n_heads=n_heads, n_kv_heads=n_kv_heads, d_ff=d_ff, max_seq_len=max_seq_len,
        )
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, n_kv_heads, d_ff) for _ in range(n_layers)])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # Tie input & output embeddings

        cos, sin = precompute_rope(d_model // n_heads, max_seq_len, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # Scale residual output projections for deep stability
        res_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.layers:
            nn.init.normal_(block.attn.o_proj.weight, mean=0.0, std=res_std)
            nn.init.normal_(block.mlp.down_proj.weight, mean=0.0, std=res_std)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            # Always compute cross-entropy in fp32
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


# Memory-mapped uint16 datasets (near-zero RAM)
en_train = np.memmap(f"{DATA_DIR}/en_train.bin", dtype=np.uint16, mode="r")
bn_train = np.memmap(f"{DATA_DIR}/bn_train.bin", dtype=np.uint16, mode="r")
en_val = np.memmap(f"{DATA_DIR}/en_val.bin", dtype=np.uint16, mode="r")
bn_val = np.memmap(f"{DATA_DIR}/bn_val.bin", dtype=np.uint16, mode="r")

def get_batch(mem_arr, batch_size, seq_len, device="cuda"):
    ix = torch.randint(0, len(mem_arr) - seq_len - 1, (batch_size,))
    x = torch.stack([torch.from_numpy(mem_arr[i : i + seq_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(mem_arr[i + 1 : i + 1 + seq_len].astype(np.int64)) for i in ix])
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

def get_bilingual_batch(batch_size, seq_len, device="cuda"):
    half = batch_size // 2
    xe, ye = get_batch(en_train, half, seq_len, device)
    xb, yb = get_batch(bn_train, batch_size - half, seq_len, device)
    return torch.cat([xe, xb], dim=0), torch.cat([ye, yb], dim=0)

@torch.no_grad()
def evaluate_bpb(model, seq_len=512, eval_batches=40, batch_size=16, device="cuda"):
    model.eval()
    out = {}
    for lang, arr in [("en", en_val), ("bn", bn_val)]:
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch(arr, batch_size, seq_len, device)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                _, loss = model(x, y)
            losses.append(loss.item())
        mean_loss = float(np.mean(losses))
        tok_per_byte = META[lang]["val_tokens"] / META[lang]["val_bytes"]
        bpb = (mean_loss / math.log(2)) * tok_per_byte
        out[lang] = {"loss": round(mean_loss, 4), "bpb": round(bpb, 4)}
    model.train()
    return out

@torch.no_grad()
def sample_story(model, prompt, max_new_tokens=90, temperature=0.8, top_k=40, device="cuda"):
    model.eval()
    ids = tokenizer.encode(prompt).ids
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -512:]
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits, _ = model(idx_cond)
        logits = logits[:, -1, :].float() / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        if nxt.item() == EOS_ID:
            break
        idx = torch.cat([idx, nxt], dim=1)
    model.train()
    return tokenizer.decode(idx[0].tolist())

torch.manual_seed(42)
model = BilingualSLM(vocab_size=VOCAB_SIZE).to("cuda")
n_params = sum(p.numel() for p in model.parameters())
n_non_emb = n_params - model.tok_emb.weight.numel()
print(f"Model Ready | Total Params: {n_params/1e6:.2f}M (Non-Embedding: {n_non_emb/1e6:.2f}M)")

Model Ready | Total Params: 18.88M (Non-Embedding: 12.59M)


In [ ]:
from huggingface_hub import hf_hub_download

SEQ_LEN = 512
MICRO_BATCH = 16       # 8 En + 8 Bn per micro-batch
GRAD_ACCUM = 4         # Effective batch = 64 seqs = 32,768 tokens/step
MAX_STEPS = 6104       # 6,104 * 32,768 = 200,015,872 tokens (100M En + 100M Bn)
MAX_LR = 6e-4
MIN_LR = 6e-5
WARMUP_STEPS = 200

# Exact fork milestones: 20M, 50M, 100M, 200M tokens
MILESTONES = {610: "20M", 1526: "50M", 3052: "100M", 6104: "200M"}

# Separate weight decay (no decay on 1D norms)
decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2]
nodecay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay_params, "weight_decay": 0.1}, {"params": nodecay_params, "weight_decay": 0.0}],
    lr=MAX_LR, betas=(0.9, 0.95), eps=1e-8,
)
scaler = torch.amp.GradScaler("cuda")

def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1.0 + math.cos(math.pi * progress))

os.makedirs("checkpoints", exist_ok=True)
start_step = 0

# Auto-resume from Hugging Face if a previous Colab session disconnected
try:
    ckpt_path = hf_hub_download(repo_id=MODEL_REPO, filename="latest.pt")
    ckpt = torch.load(ckpt_path, map_location="cuda")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    start_step = ckpt["step"]
    print(f"Resumed cleanly from Hugging Face checkpoint at step {start_step}/{MAX_STEPS}!")
except Exception:
    print("Starting fresh B0 training from step 0...")

def save_and_upload(step, filename, push_full_state=False):
    path = f"checkpoints/{filename}"
    payload = {"step": step, "config": model.config, "model": model.state_dict()}
    if push_full_state:
        payload["optimizer"] = optimizer.state_dict()
        payload["scaler"] = scaler.state_dict()
    torch.save(payload, path)
    api.upload_file(path_or_fileobj=path, path_in_repo=filename, repo_id=MODEL_REPO, repo_type="model")
    print(f"  -> Uploaded {filename} to {MODEL_REPO}")

model.train()
t_start = time.time()
t_log = time.time()

for step in range(start_step + 1, MAX_STEPS + 1):
    lr = get_lr(step - 1)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    for _ in range(GRAD_ACCUM):
        x, y = get_bilingual_batch(MICRO_BATCH, SEQ_LEN)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss = model(x, y)
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if math.isnan(accum_loss) or math.isinf(accum_loss):
        raise RuntimeError(f"NaN/Inf detected at step {step}! Stopping immediately.")

    if step == start_step + 10:
        dt = (time.time() - t_start) / 10
        vram = torch.cuda.max_memory_allocated() / 1e9
        tok_s = 32768 / dt
        est_hr = (dt * (MAX_STEPS - start_step)) / 3600
        print(f"\n=== 10-STEP SAFETY CHECK === Speed: {tok_s:,.0f} tok/s ({dt:.2f}s/step) | Peak VRAM: {vram:.2f} GB | Est Total: {est_hr:.2f} hrs\n")

    if step % 100 == 0 or step == MAX_STEPS:
        dt = time.time() - t_log
        t_log = time.time()
        print(f"Step {step:4d}/{MAX_STEPS} ({step*32768/1e6:5.1f}M tok) | Loss: {accum_loss:.4f} | Grad: {grad_norm:.2f} | LR: {lr:.2e} | {dt:.1f}s/100steps")

    if step % 500 == 0:
        metrics = evaluate_bpb(model)
        print(f"  [VAL @ {step}] EN: loss={metrics['en']['loss']} bpb={metrics['en']['bpb']} | BN: loss={metrics['bn']['loss']} bpb={metrics['bn']['bpb']}")
        save_and_upload(step, "latest.pt", push_full_state=True)

    if step in MILESTONES:
        tag = MILESTONES[step]
        metrics = evaluate_bpb(model)
        print(f"\n=== MILESTONE B0_{tag} (Step {step}) ===")
        print(f"EN: {metrics['en']} | BN: {metrics['bn']}")
        print("EN Sample:", sample_story(model, "Once upon a time, a little dog"))
        print("BN Sample:", sample_story(model, "একদিন একটি ছোট্ট পাখি"))
        save_and_upload(step, f"b0_{tag}.pt", push_full_state=False)

Starting fresh B0 training from step 0...

=== 10-STEP SAFETY CHECK === Speed: 44,886 tok/s (0.73s/step) | Peak VRAM: 3.67 GB | Est Total: 1.24 hrs

Step  100/6104 (  3.3M tok) | Loss: 4.5590 | Grad: 0.71 | LR: 3.00e-04 | 64.0s/100steps
Step  200/6104 (  6.6M tok) | Loss: 3.1146 | Grad: 1.07 | LR: 6.00e-04 | 63.0s/100steps
Step  300/6104 (  9.8M tok) | Loss: 2.6408 | Grad: 0.74 | LR: 6.00e-04 | 64.4s/100steps
Step  400/6104 ( 13.1M tok) | Loss: 2.4240 | Grad: 0.72 | LR: 5.98e-04 | 64.8s/100steps
Step  500/6104 ( 16.4M tok) | Loss: 2.2377 | Grad: 0.66 | LR: 5.97e-04 | 64.6s/100steps
  [VAL @ 500] EN: loss=2.6392 bpb=0.9181 | BN: loss=1.9629 bpb=0.245


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ent/checkpoints/latest.pt:   0%|          |  553kB /  227MB            

Validating                    :           |   0%            

  -> Uploaded latest.pt to a-mondal/bilingual-slm-18m-b0
Step  600/6104 ( 19.7M tok) | Loss: 2.1354 | Grad: 0.67 | LR: 5.94e-04 | 84.8s/100steps

=== MILESTONE B0_20M (Step 610) ===
EN: {'loss': 2.5287, 'bpb': 0.8797} | BN: {'loss': 1.8645, 'bpb': 0.2327}
EN Sample: Once upon a time, a little dog named Max went on a walk. One day, Max was playing with his toy cars and he accidentally broke it. It was all gone! Max was sad because he couldn't have any toys to put him far away. So, he went to the store to buy some tools for his toys. Then, he went to the store and bought some toys. His owner took him for a walk in the woods with him. Max went outside and went
BN Sample: একদিন একটি ছোট্ট পাখি ছিল, নাম তার পিপঁড়ে। পিপঁড়ে খুবই আনন্দিত ছিল এবং সে সবসময় নতুন কিছু করার জন্য প্রস্তুত থাকত। এক সকালে, সে তার বন্ধুদের নিয়ে একটি সুন্দর বাগানে খেলতে গেল। তারা সবাই ফুলের কাছে এসে দাঁড়াল এবং বলল, "আমরা এখানে অনেক মজা করেছি!" পিপঁড়ে এবং তার বন্ধুরা ফুলের কাছে গেল। তারা ফুলের গন্ধ নিল এবং হাসি মুখ

... (71 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt/checkpoints/b0_200M.pt:   1%|          |  552kB / 75.6MB            

Validating                    :           |   0%            

  -> Uploaded b0_200M.pt to a-mondal/bilingual-slm-18m-b0


In [ ]:
from huggingface_hub import hf_hub_download

# === CHANGE ONLY THIS LINE PER RUN ===
# Options: "F3_en", "F3_bn", "F2_en", "F2_bn", "F1_en", "F1_bn", "M1", "M2"
RUN_NAME = "F3_en"

RUN_SPECS = {
    "M1":    {"init": None,          "lang": "en", "start_step": 0},
    "M2":    {"init": None,          "lang": "bn", "start_step": 0},
    "F1_en": {"init": "b0_20M.pt",   "lang": "en", "start_step": 610},
    "F1_bn": {"init": "b0_20M.pt",   "lang": "bn", "start_step": 610},
    "F2_en": {"init": "b0_50M.pt",   "lang": "en", "start_step": 1526},
    "F2_bn": {"init": "b0_50M.pt",   "lang": "bn", "start_step": 1526},
    "F3_en": {"init": "b0_100M.pt",  "lang": "en", "start_step": 3052},
    "F3_bn": {"init": "b0_100M.pt",  "lang": "bn", "start_step": 3052},
}

spec = RUN_SPECS[RUN_NAME]
TOTAL_STEPS = 6104
SEQ_LEN, MICRO_BATCH, GRAD_ACCUM = 512, 16, 4
MAX_LR, MIN_LR, WARMUP_STEPS = 6e-4, 6e-5, 200

# 1. Initialize model (from scratch or from saved B0 ancestor)
torch.manual_seed(42 if spec["lang"] == "en" else 43)
model = BilingualSLM(vocab_size=VOCAB_SIZE).to("cuda")

if spec["init"] is not None:
    anc_path = hf_hub_download(repo_id=MODEL_REPO, filename=spec["init"])
    anc = torch.load(anc_path, map_location="cuda")
    model.load_state_dict(anc["model"])
    print(f"[{RUN_NAME}] Loaded ancestor {spec['init']} (step {spec['start_step']})")
else:
    print(f"[{RUN_NAME}] Starting from scratch (step 0)")

train_arr = en_train if spec["lang"] == "en" else bn_train

decay_params = [p for p in model.parameters() if p.dim() >= 2]
nodecay_params = [p for p in model.parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay_params, "weight_decay": 0.1}, {"params": nodecay_params, "weight_decay": 0.0}],
    lr=MAX_LR, betas=(0.9, 0.95), eps=1e-8,
)
scaler = torch.amp.GradScaler("cuda")

def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1.0 + math.cos(math.pi * progress))

model.train()
t_log = time.time()

for step in range(spec["start_step"] + 1, TOTAL_STEPS + 1):
    lr = get_lr(step - 1)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    for _ in range(GRAD_ACCUM):
        x, y = get_batch(train_arr, MICRO_BATCH, SEQ_LEN)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss = model(x, y)
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if math.isnan(accum_loss) or math.isinf(accum_loss):
        raise RuntimeError(f"[{RUN_NAME}] NaN/Inf at step {step}!")

    if step % 250 == 0 or step == TOTAL_STEPS:
        dt = time.time() - t_log
        t_log = time.time()
        print(f"[{RUN_NAME}] Step {step:4d}/{TOTAL_STEPS} | Loss: {accum_loss:.4f} | Grad: {grad_norm:.2f} | LR: {lr:.2e} | {dt:.1f}s")

metrics = evaluate_bpb(model)
print(f"\n=== FINAL {RUN_NAME} (200M Total Tokens) ===")
print(f"EN: {metrics['en']} | BN: {metrics['bn']}")

out_file = f"checkpoints/{RUN_NAME.lower()}_200M.pt"
torch.save({"run": RUN_NAME, "step": TOTAL_STEPS, "config": model.config, "metrics": metrics, "model": model.state_dict()}, out_file)
api.upload_file(path_or_fileobj=out_file, path_in_repo=f"{RUN_NAME.lower()}_200M.pt", repo_id=MODEL_REPO, repo_type="model")
print(f"-> Uploaded {RUN_NAME.lower()}_200M.pt to {MODEL_REPO}")

b0_100M.pt: reconstructing file:   0%|          |  0.00B / 75.6MB            

b0_100M.pt: downloading bytes:           |  0.00B            

[F3_en] Loaded ancestor b0_100M.pt (step 3052)
[F3_en] Step 3250/6104 | Loss: 1.8671 | Grad: 0.53 | LR: 3.16e-04 | 126.6s
[F3_en] Step 3500/6104 | Loss: 1.7178 | Grad: 0.51 | LR: 2.80e-04 | 161.0s
[F3_en] Step 3750/6104 | Loss: 1.7440 | Grad: 0.53 | LR: 2.46e-04 | 160.8s
[F3_en] Step 4000/6104 | Loss: 1.6341 | Grad: 0.49 | LR: 2.12e-04 | 160.6s
[F3_en] Step 4250/6104 | Loss: 1.6627 | Grad: 0.50 | LR: 1.81e-04 | 160.5s
[F3_en] Step 4500/6104 | Loss: 1.6581 | Grad: 0.50 | LR: 1.53e-04 | 160.6s
[F3_en] Step 4750/6104 | Loss: 1.7191 | Grad: 0.51 | LR: 1.27e-04 | 160.6s
[F3_en] Step 5000/6104 | Loss: 1.7189 | Grad: 0.50 | LR: 1.05e-04 | 160.6s
[F3_en] Step 5250/6104 | Loss: 1.6085 | Grad: 0.53 | LR: 8.75e-05 | 160.8s
[F3_en] Step 5500/6104 | Loss: 1.5949 | Grad: 0.52 | LR: 7.39e-05 | 161.0s
[F3_en] Step 5750/6104 | Loss: 1.4495 | Grad: 0.52 | LR: 6.48e-05 | 161.0s
[F3_en] Step 6000/6104 | Loss: 1.6084 | Grad: 0.53 | LR: 6.04e-05 | 161.0s
[F3_en] Step 6104/6104 | Loss: 1.5952 | Grad: 0.54 | 

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...checkpoints/f3_en_200M.pt:   1%|          |  554kB / 75.6MB            

Validating                    :           |   0%            

-> Uploaded f3_en_200M.pt to a-mondal/bilingual-slm-18m-b0


In [ ]:
from huggingface_hub import hf_hub_download

# === CHANGE ONLY THIS LINE PER RUN ===
# Options: "F3_en", "F3_bn", "F2_en", "F2_bn", "F1_en", "F1_bn", "M1", "M2"
RUN_NAME = "F3_bn"

RUN_SPECS = {
    "M1":    {"init": None,          "lang": "en", "start_step": 0},
    "M2":    {"init": None,          "lang": "bn", "start_step": 0},
    "F1_en": {"init": "b0_20M.pt",   "lang": "en", "start_step": 610},
    "F1_bn": {"init": "b0_20M.pt",   "lang": "bn", "start_step": 610},
    "F2_en": {"init": "b0_50M.pt",   "lang": "en", "start_step": 1526},
    "F2_bn": {"init": "b0_50M.pt",   "lang": "bn", "start_step": 1526},
    "F3_en": {"init": "b0_100M.pt",  "lang": "en", "start_step": 3052},
    "F3_bn": {"init": "b0_100M.pt",  "lang": "bn", "start_step": 3052},
}

spec = RUN_SPECS[RUN_NAME]
TOTAL_STEPS = 6104
SEQ_LEN, MICRO_BATCH, GRAD_ACCUM = 512, 16, 4
MAX_LR, MIN_LR, WARMUP_STEPS = 6e-4, 6e-5, 200

# 1. Initialize model (from scratch or from saved B0 ancestor)
torch.manual_seed(42 if spec["lang"] == "en" else 43)
model = BilingualSLM(vocab_size=VOCAB_SIZE).to("cuda")

if spec["init"] is not None:
    anc_path = hf_hub_download(repo_id=MODEL_REPO, filename=spec["init"])
    anc = torch.load(anc_path, map_location="cuda")
    model.load_state_dict(anc["model"])
    print(f"[{RUN_NAME}] Loaded ancestor {spec['init']} (step {spec['start_step']})")
else:
    print(f"[{RUN_NAME}] Starting from scratch (step 0)")

train_arr = en_train if spec["lang"] == "en" else bn_train

decay_params = [p for p in model.parameters() if p.dim() >= 2]
nodecay_params = [p for p in model.parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay_params, "weight_decay": 0.1}, {"params": nodecay_params, "weight_decay": 0.0}],
    lr=MAX_LR, betas=(0.9, 0.95), eps=1e-8,
)
scaler = torch.amp.GradScaler("cuda")

def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1.0 + math.cos(math.pi * progress))

model.train()
t_log = time.time()

for step in range(spec["start_step"] + 1, TOTAL_STEPS + 1):
    lr = get_lr(step - 1)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    for _ in range(GRAD_ACCUM):
        x, y = get_batch(train_arr, MICRO_BATCH, SEQ_LEN)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            _, loss = model(x, y)
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if math.isnan(accum_loss) or math.isinf(accum_loss):
        raise RuntimeError(f"[{RUN_NAME}] NaN/Inf at step {step}!")

    if step % 250 == 0 or step == TOTAL_STEPS:
        dt = time.time() - t_log
        t_log = time.time()
        print(f"[{RUN_NAME}] Step {step:4d}/{TOTAL_STEPS} | Loss: {accum_loss:.4f} | Grad: {grad_norm:.2f} | LR: {lr:.2e} | {dt:.1f}s")

metrics = evaluate_bpb(model)
print(f"\n=== FINAL {RUN_NAME} (200M Total Tokens) ===")
print(f"EN: {metrics['en']} | BN: {metrics['bn']}")

out_file = f"checkpoints/{RUN_NAME.lower()}_200M.pt"
torch.save({"run": RUN_NAME, "step": TOTAL_STEPS, "config": model.config, "metrics": metrics, "model": model.state_dict()}, out_file)
api.upload_file(path_or_fileobj=out_file, path_in_repo=f"{RUN_NAME.lower()}_200M.pt", repo_id=MODEL_REPO, repo_type="model")
print(f"-> Uploaded {RUN_NAME.lower()}_200M.pt to {MODEL_REPO}")

[F3_bn] Loaded ancestor b0_100M.pt (step 3052)
[F3_bn] Step 3250/6104 | Loss: 1.1753 | Grad: 0.46 | LR: 3.16e-04 | 128.1s
[F3_bn] Step 3500/6104 | Loss: 1.1814 | Grad: 0.48 | LR: 2.80e-04 | 160.6s
[F3_bn] Step 3750/6104 | Loss: 1.1408 | Grad: 0.47 | LR: 2.46e-04 | 160.9s
[F3_bn] Step 4000/6104 | Loss: 1.1800 | Grad: 0.49 | LR: 2.12e-04 | 160.9s
[F3_bn] Step 4250/6104 | Loss: 1.1089 | Grad: 0.49 | LR: 1.81e-04 | 160.8s
[F3_bn] Step 4500/6104 | Loss: 1.1055 | Grad: 0.48 | LR: 1.53e-04 | 160.8s
[F3_bn] Step 4750/6104 | Loss: 1.0468 | Grad: 0.49 | LR: 1.27e-04 | 160.8s
[F3_bn] Step 5000/6104 | Loss: 1.0056 | Grad: 0.47 | LR: 1.05e-04 | 160.8s
[F3_bn] Step 5250/6104 | Loss: 1.0467 | Grad: 0.49 | LR: 8.75e-05 | 160.9s
[F3_bn] Step 5500/6104 | Loss: 1.0407 | Grad: 0.49 | LR: 7.39e-05 | 161.1s
[F3_bn] Step 5750/6104 | Loss: 1.0354 | Grad: 0.50 | LR: 6.48e-05 | 160.9s
[F3_bn] Step 6000/6104 | Loss: 0.9983 | Grad: 0.50 | LR: 6.04e-05 | 160.8s
[F3_bn] Step 6104/6104 | Loss: 0.9860 | Grad: 0.50 | 

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...checkpoints/f3_bn_200M.pt:   1%|          |  556kB / 75.6MB            

Validating                    :           |   0%            

-> Uploaded f3_bn_200M.pt to a-mondal/bilingual-slm-18m-b0


In [ ]:
import torch
import numpy as np
from huggingface_hub import hf_hub_download

# 1. Load F3_en and F3_bn weights
sd_en = torch.load(hf_hub_download(MODEL_REPO, "f3_en_200M.pt"), map_location="cuda")["model"]
sd_bn = torch.load(hf_hub_download(MODEL_REPO, "f3_bn_200M.pt"), map_location="cuda")["model"]

def merge_linear(sd1, sd2, alpha):
    return {k: (1.0 - alpha) * sd1[k].float() + alpha * sd2[k].float() for k in sd1}

def merge_slerp(sd1, sd2, alpha, eps=1e-8):
    out = {}
    for k in sd1:
        v1, v2 = sd1[k].float(), sd2[k].float()
        if v1.ndim < 2:  # 1D norms -> linear
            out[k] = (1.0 - alpha) * v1 + alpha * v2
            continue
        f1, f2 = v1.flatten(), v2.flatten()
        n1, n2 = f1.norm(), f2.norm()
        dot = torch.clamp(torch.dot(f1 / (n1 + eps), f2 / (n2 + eps)), -1.0, 1.0)
        omega = torch.acos(dot)
        if omega.abs() < 1e-4:
            out[k] = (1.0 - alpha) * v1 + alpha * v2
        else:
            so = torch.sin(omega)
            w1 = torch.sin((1.0 - alpha) * omega) / so
            w2 = torch.sin(alpha * omega) / so
            # Interpolate direction spherically, scale by linearly interpolated norm
            merged_dir = w1 * (v1 / (n1 + eps)) + w2 * (v2 / (n2 + eps))
            out[k] = merged_dir * ((1.0 - alpha) * n1 + alpha * n2)
    return out

# 2. Sweep alpha for Linear & SLERP
eval_model = BilingualSLM(vocab_size=VOCAB_SIZE).to("cuda")

for name, fn in [("LINEAR", merge_linear), ("SLERP", merge_slerp)]:
    print(f"\n================ {name} MERGE (F3_en <-> F3_bn) ================")
    print(f"{'Alpha (BN)':>10s} | {'EN Loss':>8s} {'EN BPB':>8s} | {'BN Loss':>8s} {'BN BPB':>8s}")
    print("-" * 55)
    for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
        eval_model.load_state_dict(fn(sd_en, sd_bn, alpha))
        m = evaluate_bpb(eval_model)
        print(f"{alpha:10.2f} | {m['en']['loss']:8.4f} {m['en']['bpb']:8.4f} | {m['bn']['loss']:8.4f} {m['bn']['bpb']:8.4f}")

# 3. Sample stories from 50/50 SLERP Merged Model
eval_model.load_state_dict(merge_slerp(sd_en, sd_bn, 0.5))
print("\n=== 50/50 SLERP MERGED SAMPLES (ZERO RETRAINING) ===")
print("EN:", sample_story(eval_model, "Once upon a time, a little dog"))
print("BN:", sample_story(eval_model, "একদিন একটি ছোট্ট পাখি"))

# 4. Layer-wise DoRA Weight Decomposition (F3_en vs F3_bn)
print("\n=== LAYER-WISE DoRA DIVERGENCE (F3_en vs F3_bn) ===")
print(f"{'Layer':>5s} | {'Attn Dir Dist':>13s} {'Attn Mag Diff':>13s} | {'MLP Dir Dist':>12s} {'MLP Mag Diff':>12s}")
print("-" * 68)
for l in range(8):
    def layer_stats(keys):
        d_list, m_list = [], []
        for sub in keys:
            w1 = sd_en[f"layers.{l}.{sub}.weight"].float()
            w2 = sd_bn[f"layers.{l}.{sub}.weight"].float()
            # Column-wise L2 norm (magnitude) and cosine distance (1 - cos_sim)
            m1, m2 = w1.norm(dim=0), w2.norm(dim=0)
            cos = torch.nn.functional.cosine_similarity(w1, w2, dim=0)
            d_list.append((1.0 - cos).mean().item())
            m_list.append((m1 - m2).abs().mean().item())
        return np.mean(d_list), np.mean(m_list)

    ad, am = layer_stats(["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj"])
    md, mm = layer_stats(["mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"])
    print(f"  L{l:02d} | {ad:13.4f} {am:13.4f} | {md:12.4f} {mm:12.4f}")

f3_en_200M.pt: reconstructing file:   0%|          |  0.00B / 75.6MB            

f3_en_200M.pt: downloading bytes:           |  0.00B            

f3_bn_200M.pt: reconstructing file:   0%|          |  0.00B / 75.6MB            

f3_bn_200M.pt: downloading bytes:           |  0.00B            


================ LINEAR MERGE (F3_en <-> F3_bn) ================
Alpha (BN) |  EN Loss   EN BPB |  BN Loss   BN BPB
-------------------------------------------------------
      0.00 |   1.6580   0.5768 |   9.2971   1.1602
      0.25 |   1.8809   0.6543 |   7.7248   0.9640
      0.50 |   2.7941   0.9720 |   4.6253   0.5772
      0.75 |   4.5997   1.6001 |   1.4199   0.1772
      1.00 |   6.5774   2.2881 |   1.0948   0.1366

================ SLERP MERGE (F3_en <-> F3_bn) ================
Alpha (BN) |  EN Loss   EN BPB |  BN Loss   BN BPB
-------------------------------------------------------
      0.00 |   1.6717   0.5815 |   9.2868   1.1589
      0.25 |   1.9870   0.6912 |   7.1130   0.8877
      0.50 |   2.7576   0.9593 |   3.3723   0.4208
      0.75 |   4.3073   1.4984 |   1.3856   0.1729
      1.00 |   6.5768   2.2879 |   1.1285   0.1408

=== 50/50 SLERP MERGED SAMPLES (ZERO RETRAINING) ===
EN: Once upon a time, a little dog, and a few. He was a bit scared, but he was happy to see